# Unit 2 — Custom function tools

**The agent decides when to call tools. You never call them yourself.** That's the shift.

In [ ]:
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

from typing import Annotated
from pydantic import Field
from agent_framework.openai import OpenAIChatCompletionClient

## Define two tools as plain Python functions

The `Annotated[type, Field(description=...)]` pattern gives the model both *what* each arg is and *what it means*. Without it, tool use gets unreliable.

In [ ]:
def get_weather(
    city: Annotated[str, Field(description="The city to look up, e.g. 'Berlin'.")],
) -> str:
    """Return the current weather for a city."""
    faked = {"berlin": "12°C, overcast", "vienna": "15°C, sunny", "munich": "10°C, light rain"}
    return faked.get(city.lower(), f"weather data unavailable for {city}")

def get_time(
    timezone: Annotated[str, Field(description="IANA timezone, e.g. 'Europe/Berlin'.")],
) -> str:
    """Return the current local time for a timezone."""
    from datetime import datetime
    return f"It is currently {datetime.now().strftime('%H:%M')} in {timezone}."

## Attach them to an agent

`tools=[...]` — the agent now knows these functions exist and when to use them.

In [ ]:
agent = OpenAIChatCompletionClient().as_agent(
    name="TravelHelper",
    instructions="You help travelers. Use the provided tools to look up weather and time. Keep answers concise.",
    tools=[get_weather, get_time],
)

In [ ]:
result = await agent.run("What's the weather in Berlin and what time is it there?")
print(result.text)

## Experiments

Try a prompt that needs **no** tools — the agent shouldn't call them.

In [ ]:
result = await agent.run("Tell me a one-line travel joke.")
print(result.text)

Now something ambiguous — 'What's it like in Munich?' The agent has to guess your intent.

In [ ]:
result = await agent.run("What's it like in Munich?")
print(result.text)